# Download & Setup Dataset

In [ ]:
# import os
# os.environ["KAGGLE_USERNAME"]=""
# os.environ["KAGGLE_KEY"]=""

# !kaggle datasets download -d manjilkarki/deepfake-and-real-images
# !unzip -q deepfake-and-real-images.zip

Dataset URL: https://www.kaggle.com/datasets/manjilkarki/deepfake-and-real-images
License(s): unknown
 99% 1.67G/1.68G [00:17<00:00, 218MB/s]
100% 1.68G/1.68G [00:17<00:00, 106MB/s]


In [ ]:
import os

data_dir="Dataset"

print(f"Contents of {data_dir}:")
if os.path.exists(data_dir):
    print(os.listdir(data_dir))
else:
    print(f"Error: Directory {data_dir} does not exist.")


Contents of Dataset:
['Test', 'Train', 'Validation']


# Model Architecture

## Data Prepration

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = (192, 192)
BATCH_SIZE = 16
AUTOTUNE=tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir + "/Train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir + "/Validation",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

preprocess=tf.keras.applications.efficientnet.preprocess_input

train_ds = train_ds.map(
    lambda x,y: (preprocess(x),y),
    num_parallel_calls=AUTOTUNE
)

val_ds = val_ds.map(
    lambda x,y: (preprocess(x),y),
    num_parallel_calls=AUTOTUNE
)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

Found 140002 files belonging to 2 classes.
Found 39428 files belonging to 2 classes.


# Model Design

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

base_model = tf.keras.applications.EfficientNetB0(
    input_shape=(192,192,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False


# model = models.Sequential([
#     data_augmentation,
#     base_model,
#     layers.GlobalAveragePooling2D(),
#     layers.Dropout(0.5),
#     layers.Dense(1, activation='sigmoid')
# ])

inputs = tf.keras.Input(shape=(192, 192, 3))

x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs)


model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-5),
    loss='binary_crossentropy',
    metrics=['accuracy',tf.keras.metrics.AUC()]
)


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
# Test Dataset

test_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir + "/Test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)
test_ds = test_ds.map(
    lambda x,y: (preprocess(x),y),
    num_parallel_calls=AUTOTUNE
)
test_ds = test_ds.prefetch(AUTOTUNE)

Found 10905 files belonging to 2 classes.


# Training

In [ ]:
!mkdir -p /content/models
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
        "/content/models/best_model.h5",
        save_best_only=True
    )
]

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks,
    verbose=2
)
# model.save("/content/deepfake_model.keras")
model.save_weights("deepfake_v1.weights.h5")

Epoch 1/15


8751/8751 - 534s - 61ms/step - accuracy: 0.8719 - auc: 0.9481 - loss: 0.2901 - val_accuracy: 0.8951 - val_auc: 0.9756 - val_loss: 0.2613
Epoch 2/15


In [ ]:
# Log
import matplotlib.pyplot as plt

# Accuracy Plot
plt.figure(figsize=(8,5))

plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")

plt.legend()
plt.savefig("/content/accuracy_plot.png")

plt.show()

# Loss Plot
plt.figure(figsize=(8,5))

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")

plt.legend()
plt.savefig("/content/loss_plot.png")

plt.show()

print("Final Training Accuracy:",
      history.history['accuracy'][-1])

print("Final Validation Accuracy:",
      history.history['val_accuracy'][-1])

print("Final Training Loss:",
      history.history['loss'][-1])

print("Final Validation Loss:",
      history.history['val_loss'][-1])

In [ ]:
# Download Model & Charts

from google.colab import files

files.download("/content/accuracy_plot.png")
files.download("/content/loss_plot.png")
# files.download("/content/deepfake_model.keras")
files.download("/content/deepfake_v1.weights.h5")

# Infrence

In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model("/content/deepfake_model.keras")

import numpy as np
from tensorflow.keras.preprocessing import image

IMG_SIZE = (192,192)

preprocess = tf.keras.applications.efficientnet.preprocess_input

def preprocess_image(img_path):

    img = image.load_img(img_path, target_size=IMG_SIZE)

    img_array = image.img_to_array(img)

    img_array = np.expand_dims(img_array, axis=0)

    img_array = preprocess(img_array)

    return img_array


In [ ]:
import matplotlib.pyplot as plt

def predict_and_show(img_path):

    img = image.load_img(img_path, target_size=IMG_SIZE)

    processed = preprocess_image(img_path)

    pred = model.predict(processed)[0][0]

    label = "Fake" if pred > 0.5 else "Real"

    plt.imshow(img)
    plt.axis("off")

    plt.title(f"{label} ({pred:.2f})")

    plt.show()

In [ ]:
predict_and_show("/content/test2.jpg")